In [1]:
import pandas as pd
import sys
sys.path.append("..")
from app.features import make_features
from app.backtest import walk_forward_splits

In [2]:
y = pd.read_parquet("../data/processed/hourly_jan2026.parquet")["trips"]
X = make_features(y).dropna()
y1 = y.loc[X.index]
folds = list(walk_forward_splits(X, 24*14, 24, 24, "expanding"))

In [3]:
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_absolute_percentage_error

In [4]:
rows = []
for tr, te in folds:
    model = LGBMRegressor(
    num_leaves=5,
    min_child_samples=7,
    n_estimators=50,
    random_state=42,
    )
    
    model.fit(X.loc[tr], y1.loc[tr])
    y_pred = pd.Series(model.predict(X.loc[te]), index=te)
    y_true = y1.loc[te]
    rows.append({"MAE": mean_absolute_error(y_true, y_pred), "MAPE": mean_absolute_percentage_error(y_true, y_pred)})


res = pd.DataFrame(rows)
res.agg(["mean", "median", "std"])    

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000182 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 259
[LightGBM] [Info] Number of data points in the train set: 336, number of used features: 5
[LightGBM] [Info] Start training from score 5223.181548
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000101 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 275
[LightGBM] [Info] Number of data points in the train set: 360, number of used features: 5
[LightGBM] [Info] Start training from score 5253.247222
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000063 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 291
[LightGBM] [Info] Number of data points in the train set: 384, number of used features: 5
[LightGBM] [Info] Start training 

,MAE,MAPE
mean,1106.945989,0.495382
median,638.150455,0.156454
std,1000.400769,0.777676


In [11]:
res

,MAE,MAPE
0,391.018814,0.183923
1,599.827054,0.112266
2,676.473855,0.107799
3,3534.532696,2.447848
4,1743.306399,1.306015
5,569.173563,0.127113
6,454.643087,0.160778
7,541.391829,0.152130
8,705.122927,0.097746
9,1853.969661,0.258203


In [10]:
for i, m in enumerate(res):
    print(i, round(m, 3))


TypeError: type str doesn't define __round__ method

In [ ]:
pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)

In [ ]:
assert "y" not in X.columns
print(X.columns.tolist()) 